Universidad Torcuato Di Tella

Licenciatura en Tecnología Digital\
**Tecnología Digital VI: Inteligencia Artificial**

Importamos librerias

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt

Definimos una transformación simple para imágenes RGB

In [2]:
transform = transforms.Compose([
    transforms.ToTensor() #ToTensor ya normaliza a valores en [0,1]
])

Para esta prueba vamos a utilizar el dataset MNIST

In [3]:
train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
targets_ = train_set.targets

train_idx, val_idx = train_test_split(np.arange(len(targets_)), test_size = 0.2, stratify = targets_)
train_sampler = torch.utils.data.SubsetRandomSampler(train_idx)
val_sampler = torch.utils.data.SubsetRandomSampler(val_idx)

train_loader = DataLoader(train_set, sampler=train_sampler, batch_size=64)
val_loader = DataLoader(train_set, sampler=val_sampler, batch_size=64)
test_loader = DataLoader(test_set, batch_size=64)

100%|██████████| 9.91M/9.91M [00:00<00:00, 12.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 336kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.16MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 16.5MB/s]


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Arquitectura del Autoencoder

In [5]:
class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()

        # Encoder
        self.enc_conv1 = nn.Conv2d(1, 16, 3, padding=1, stride=1)  # entrada: 28x28x1 salida: 28x28x16
        self.pool1 = nn.MaxPool2d(2, 2)  # salida: 14x14x16
        self.enc_conv2 = nn.Conv2d(16, 32, 3, padding=1, stride=1)  # salida: 14x14x32
        self.pool2 = nn.MaxPool2d(2, 2)  # salida: 7x7x32
        self.enc_conv3 = nn.Conv2d(32, 32, 3, padding=1, stride=1)  # salida: 7x7x32

        # Decoder
        self.dec_conv1 = nn.ConvTranspose2d(32, 32, 2, stride=2)  # entrada: 7x7x32 salida: 14x14x32
        self.dec_conv2 = nn.ConvTranspose2d(32, 16, 2, stride=2)  # entrada: 14x14x32 salida: 28x28x16
        self.dec_conv3 = nn.ConvTranspose2d(16, 1, 3, padding=1)  # entrada: 28x28x16 salida: 28x28x1

    def forward_encoder(self, x):
        x = F.relu(self.enc_conv1(x))
        x = self.pool1(x)
        x = F.relu(self.enc_conv2(x))
        x = self.pool2(x)
        x = F.relu(self.enc_conv3(x))
        return x

    def forward_decoder(self, x):
        x = F.relu(self.dec_conv1(x))
        x = F.relu(self.dec_conv2(x))
        x = torch.sigmoid(self.dec_conv3(x))
        return x

    def forward(self, x):
        x = self.forward_encoder(x)
        x = self.forward_decoder(x)
        return x

autoencoder = Autoencoder().to(device)

Definimos función de pérdida y optimizador

In [6]:
criterion = nn.MSELoss().to(device)
optimizer = torch.optim.SGD(autoencoder.parameters(), lr=0.01, momentum=0.9)

Definimos la función que agrega ruido a las imágenes

In [7]:
def add_gaussian_noise(images, mean=0., std=0.1):
    noise = torch.randn(images.size()) * std + mean
    noise = noise.to(device)
    noisy_images = images + noise
    noisy_images = torch.clamp(noisy_images, 0., 1.)  # Asegúrate de que los valores estén entre 0 y 1
    return noisy_images

Entrenamos

In [8]:
num_epochs = 20


for epoch in range(num_epochs):
  train_loss = []
  val_loss = []


  for data in train_loader:
    imgs, labels = data
    imgs = imgs.to(device)
    noisy_imgs = add_gaussian_noise(imgs)  # Agrega ruido gaussiano a las imágenes
    outputs = autoencoder(noisy_imgs)
    loss = criterion(outputs, imgs) # Comparar la imagen reconstruida con la original

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    train_loss.append(loss.item() * imgs.size(0))


  with torch.no_grad():
    for data_val in val_loader:
      imgs_val, labels_val = data_val
      imgs_val = imgs_val.to(device)
      noisy_imgs_val = add_gaussian_noise(imgs_val)  # Agrega ruido gaussiano a las imágenes
      outputs_val = autoencoder(noisy_imgs_val)
      loss_val = criterion(outputs_val, imgs_val) # Comparar la imagen reconstruida con la original

      val_loss.append(loss_val.item() * imgs_val.size(0))



  print(f'Epoch: {epoch+1}, Train Loss: {np.mean(train_loss):.4f}, Val Loss: {np.mean(val_loss):.4f}')

Observemos los resultados con algunas imágenes de train

In [ ]:
train_iter = iter(train_loader)
imgs, labels = next(train_iter)
imgs = imgs.to(device)
noisy_imgs = add_gaussian_noise(imgs)  # Agrega ruido gaussiano a las imágenes

with torch.no_grad():
    outputs = autoencoder(noisy_imgs)

imgs = imgs.cpu().numpy()
noisy_imgs = noisy_imgs.cpu().numpy()
outputs = outputs.cpu().numpy()

num_imgs = 5
for i in range(num_imgs):
    original_image = np.transpose(noisy_imgs[i], (1, 2, 0))
    reconstructed_image = np.transpose(outputs[i], (1, 2, 0))


    # Mostrar la imagen original
    plt.figure(figsize=(9, 2))
    plt.subplot(1, 2, 1)
    plt.imshow(original_image, interpolation='none')
    plt.title('Original Image')

    # Mostrar la imagen reconstruida
    plt.subplot(1, 2, 2)
    plt.imshow(reconstructed_image, interpolation='none')
    plt.title('Reconstructed Image')

    plt.show()


Observemos los resultados con algunas imágenes de test

In [ ]:
test_iter = iter(test_loader)
imgs, labels = next(test_iter)
imgs = imgs.to(device)
noisy_imgs = add_gaussian_noise(imgs)  # Agrega ruido gaussiano a las imágenes

with torch.no_grad():
    outputs = autoencoder(noisy_imgs)

imgs = imgs.cpu().numpy()
noisy_imgs = noisy_imgs.cpu().numpy()
outputs = outputs.cpu().numpy()

num_imgs = 5
for i in range(num_imgs):
    original_image = np.transpose(noisy_imgs[i], (1, 2, 0))
    reconstructed_image = np.transpose(outputs[i], (1, 2, 0))


    # Mostrar la imagen original
    plt.figure(figsize=(9, 2))
    plt.subplot(1, 2, 1)
    plt.imshow(original_image, interpolation='none')
    plt.title('Original Image')

    # Mostrar la imagen reconstruida
    plt.subplot(1, 2, 2)
    plt.imshow(reconstructed_image, interpolation='none')
    plt.title('Reconstructed Image')

    plt.show()